# Data Segmentation & Statistical Analysis
This notebook performs data cleaning, exploratory analysis, distribution checks, crosstabulation/contingency testing, and trend visualizations for text segmentation data.

# 0. Environment Setup & Data Preparation
## 0.1 Import Libraries
We import pandas and numpy for data manipulation, matplotlib, seaborn, and plotly for plotting, and scipy.stats for statistical hypothesis testing (Chi-square and Fisher's exact tests).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import chisquare, chi2_contingency, fisher_exact

# Set default styling for seaborn
sns.set_theme(style="whitegrid")

## 0.2 Load the Dataset
Load the CSV files containing text segments, metadata, as well as segmentation and classification labels (df).

In [ ]:
df = pd.read_csv("data/shenbao_full.csv")
print(f"Total rows loaded: {len(df)}")
df.head(5)


## 0.3 Error Diagnostics & Data Cleaning
Perform diagnostic counts, extract four-digit year values from the date column (formatted as YYYYMMDD), and standardize boolean/categorical variables.

In [ ]:
# Diagnostics
print("Processing Error Counts:")
print(df['processing_error'].value_counts(dropna=False))

print("\nSegmentation Label Counts:")
print(df['segmentation_label'].value_counts(dropna=False).head(18))

print("\nNon-verbatim Segments Count:")
print(df['segment_verbatim'].value_counts(dropna=False).head(10))

print("\nNon-verbatim Documents Count:")
print(df.drop_duplicates(['doc_id', 'segment_verbatim'])['segment_verbatim'].value_counts(dropna=False).head(10))

# Clean and transform columns
df['year'] = pd.to_numeric(df['date'].astype(str).str.slice(0, 4), errors='coerce').astype('Int64')

print("\nData Overview:") 
summary_stats = pd.DataFrame({
    'n_segments': [len(df)],
    'n_documents': [df['doc_id'].nunique()],
    'min_year': [df['year'].min()],
    'max_year': [df['year'].max()]
    })
print(summary_stats)


In [ ]:
df.to_csv("segments_full_combined.csv", index=False)

## 0.4 Unnest Multi-label topic_type Values
Split semicolon-delimited values in topic_type (e.g., "biography; organization") into individual rows. The main DataFrame df remains unchanged for non-topic-specific analyses.

In [ ]:
# Explode multi-label topics into df_topic
df_topic = df.copy()
df_topic['topic_type'] = df_topic['topic_type'].astype(str).str.split(r'\s*;\s*')
df_topic = df_topic.explode('topic_type')
df_topic['topic_type'] = df_topic['topic_type'].str.strip()

# Remove nulls and empty strings
df_topic = df_topic[~df_topic['topic_type'].isna() & (df_topic['topic_type'] != "") & (df_topic['topic_type'] != "nan")]

print("\nTopic Type Segments Count:")
print(df_topic['topic_type'].value_counts(dropna=False).head(10))

# 1. Distribution Analyses
## 1.0 Segment Label Distribution Across Documents
Calculate document frequencies and percentages across segmentation labels, export summary table, and create a horizontal bar chart.

In [ ]:
total_docs = df['doc_id'].nunique()

segment_label = (
    df.groupby('segmentation_label')['doc_id']
    .nunique()
    .reset_index(name='n_documents')
    .assign(percent_of_documents=lambda x: (x['n_documents'] / total_docs) * 100)
    .sort_values(by='n_documents', ascending=False)
)

print(segment_label)
segment_label.to_csv("tables/segment_label.csv", index=False)

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(
    data=segment_label,
    y='segmentation_label',
    x='n_documents',
    color='steelblue'
)
plt.title("Distribution of Documents by Segmentation Label")
plt.xlabel("Number of documents")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 1.1 Relevance Type Distributions
Filter out missing values in relevance_to_original_topic, evaluate distributions at both segment and document levels, and merge outputs.

In [ ]:
# Filter non-null relevance
df_relevance = df[df['relevance_to_original_topic'].notna()].copy()
df_no_na_docs = df_relevance['doc_id'].nunique()

# Segment-level
relevance_segments = (
    df_relevance['relevance_to_original_topic']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'relevance_to_original_topic', 'count': 'n_segments'})
    .assign(percent=lambda x: (x['n_segments'] / x['n_segments'].sum()) * 100)
)

# Document-level
relevance_documents = (
    df_relevance.groupby('relevance_to_original_topic')['doc_id']
    .nunique()
    .reset_index(name='n_documents')
    .assign(percent_of_documents=lambda x: (x['n_documents'] / df_no_na_docs) * 100)
    .sort_values(by='n_documents', ascending=False)
)

# Merge & Save
relevance_distribution = pd.merge(relevance_segments, relevance_documents, on='relevance_to_original_topic')
print(relevance_distribution)
relevance_distribution.to_csv("tables/relevance_distribution.csv", index=False)

# Visualization
plt.figure(figsize=(8, 4))
sns.barplot(data=relevance_segments, x='relevance_to_original_topic', y='n_segments', color='steelblue')
plt.title("Distribution of Segments by Relevance Type")
plt.xlabel("")
plt.ylabel("Number of segments")
plt.tight_layout()
plt.show()

## 1.2 Topic Type Distributions (Segment & Document Level)
Calculate segment-level and document-level distributions for each topic, join them into a unified table, export to CSV, and visualize.

In [ ]:
# Segment-level topic counts
topic_segments = (
    df_topic['topic_type']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'topic_type', 'count': 'n_segments'})
    .assign(percent=lambda x: (x['n_segments'] / x['n_segments'].sum()) * 100)
)

# Document-level topic counts
topic_documents = (
    df_topic.groupby('topic_type')['doc_id']
    .nunique()
    .reset_index(name='n_documents')
    .assign(percent_of_documents=lambda x: (x['n_documents'] / total_docs) * 100)
    .sort_values(by='n_documents', ascending=False)
)

# Combined distribution table
topic_distribution = pd.merge(topic_segments, topic_documents, on='topic_type')
print(topic_distribution)
topic_distribution.to_csv("tables/topic_distribution.csv", index=False)

# Plot: Segment Level
plt.figure(figsize=(10, 5))
sns.barplot(data=topic_segments, y='topic_type', x='n_segments', color='steelblue')
plt.title("Distribution of Segments by Topic Type")
plt.xlabel("Number of segments")
plt.ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
# Plot: Document Level
plt.figure(figsize=(10, 7))
sns.barplot(data=topic_documents, y='topic_type', x='n_documents', color='steelblue')
plt.title("Distribution of Documents by Topic Type")
plt.suptitle("A document can appear in more than one topic", fontsize=9, y=0.96, x=0.55)
plt.xlabel("Number of documents")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 1.3 Gender Mention Distribution

Calculate overall gender representation across segments and documents and export metrics.

In [ ]:
df_gender = df[df['gender_mentioned'].notna()].copy()

# Segment level
gender_segments = (
    df_gender['gender_mentioned']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'gender_mentioned', 'count': 'n_segments'})
    .assign(percent=lambda x: (x['n_segments'] / x['n_segments'].sum()) * 100)
)

# Document level
gender_documents = (
    df_gender.groupby('gender_mentioned')['doc_id']
    .nunique()
    .reset_index(name='n_documents')
    .assign(percent_of_documents=lambda x: (x['n_documents'] / total_docs) * 100)
    .sort_values(by='n_documents', ascending=False)
)

# Merge & Export
gender_distribution = pd.merge(gender_segments, gender_documents, on='gender_mentioned')
print(gender_distribution)
gender_distribution.to_csv("gender_distribution.csv", index=False)


# Visualization
plt.figure(figsize=(8, 4))
sns.barplot(data=gender_segments, x='gender_mentioned', y='n_segments', color='steelblue')
plt.title("Distribution of Segments by Gender Mention")
plt.xlabel("")
plt.ylabel("Number of segments")
plt.tight_layout()
plt.show()

# 2. Temporal Trends
## 2.1 Relevance Over Time
Track yearly composition of relevance categories using stacked counts, 100% stacked proportions, and an interactive Plotly line chart.

In [ ]:
# Aggregation
relevance_year = (
    df.dropna(subset=['year', 'relevance_to_original_topic'])
    .groupby(['year', 'relevance_to_original_topic'])
    .size()
    .reset_index(name='n')
)
relevance_year['proportion'] = relevance_year.groupby('year')['n'].transform(lambda x: (x / x.sum()) * 100)

# Stacked Bar Chart (Absolute Counts)
pivot_counts = relevance_year.pivot(index='year', columns='relevance_to_original_topic', values='n').fillna(0)
pivot_counts.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='tab10')
plt.title("Relevance to Original Topic Over Time")
plt.xlabel("Year")
plt.ylabel("Number of segments")
plt.legend(title="Relevance")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# 100% Stacked Bar Chart (Proportions)
pivot_props = relevance_year.pivot(index='year', columns='relevance_to_original_topic', values='proportion').fillna(0)
pivot_props.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='tab10')
plt.title("Distribution of Relevance Over Time")
plt.xlabel("Year")
plt.ylabel("Share of segments (%)")
plt.legend(title="Relevance")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Interactive Stacked Bar Chart using Plotly

import plotly.express as px

# Start from your aggregated topic_year table.
plot_relevance = relevance_year.copy()
plot_relevance["year"] = pd.to_numeric(plot_relevance["year"]).astype(int)
plot_relevance = plot_relevance.sort_values(["year", "relevance_to_original_topic"])

# Use consistent topic colors and ordering across both charts.
relevance_types = plot_relevance["relevance_to_original_topic"].drop_duplicates().tolist()
color_map = {
    "RELEVANT_US": "#636EFA",      # Blue
    "RELEVANT_NON_US": "#EF553B",  # Red
}




def make_interactive_chart(value, title, ylabel):
    fig = px.bar(
        plot_relevance,
        x="year",
        y=value,
        color="relevance_to_original_topic",
        barmode="stack",
        title=title,
        labels={
            "year": "Year",
            "relevance_to_original_topic": "Relevance to Original Topic",
            "n": "Number of segments",
            "proportion": "Share of segments (%)",
        },
        category_orders={"relevance_to_original_topic": relevance_types},
        color_discrete_map=color_map,
        hover_data={
            "year": True,
            "n": ":,.0f",
            "proportion": ":.1f",
        },
        template="plotly_white",
        height=500,
    )

    # Numeric year axis: show a tick every two calendar years.
    fig.update_xaxes(
        tickmode="linear",
        tick0=int(plot_relevance["year"].min()),
        dtick=2,
        tickformat="d",
        tickangle=-45,
        tickfont=dict(size=10),
        title="Year",
    )

    fig.update_yaxes(title=ylabel)
    fig.update_layout(
        legend_title_text="Relevance",
        bargap=0.15,
    )

    if value == "proportion":
        fig.update_yaxes(range=[0, 100], ticksuffix="%")

    return fig


fig_relevance_counts = make_interactive_chart(
    "n",
    "Relevance Distribution Over Time",
    "Number of segments",
)

fig_relevance_counts.show()


In [ ]:
fig_relevance_props = make_interactive_chart(
    "proportion",
    "Distribution of Relevance Categories Over Time",
    "Share of segments (%)",
)

fig_relevance_props.show()

In [ ]:
# Interactive Plotly Line Chart
fig_relevance_line = px.line(
    relevance_year,
    x='year',
    y='proportion',
    color='relevance_to_original_topic',
    line_dash='relevance_to_original_topic',
    markers=True,
    title="Changing Relevance Composition Over Time",
    labels={'proportion': 'Share of segments (%)', 'year': 'Year', 'relevance_to_original_topic': 'Relevance'}
)
fig_relevance_line.show()

In [ ]:
# Optional: save standalone interactive files to open in a browser.
fig_relevance_counts.write_html("figures/relevance_year_counts_interactive.html")
fig_relevance_props.write_html("figures/relevance_year_props_interactive.html")
fig_relevance_line.write_html("figures/relevance_composition_interactive.html")

## 2.2 Topic Type Over Time
Evaluate how topic coverage evolves year over year using stacked bars and interactive time-series plots.

In [ ]:
import pandas as pd
import plotly.express as px

# Aggregation
topic_year = (
    df_topic.dropna(subset=['year', 'topic_type'])
    .groupby(['year', 'topic_type'])
    .size()
    .reset_index(name='n')
)
topic_year['proportion'] = topic_year.groupby('year')['n'].transform(lambda x: (x / x.sum()) * 100)

# Start from your aggregated topic_year table.
plot_data = topic_year.copy()
plot_data["year"] = pd.to_numeric(plot_data["year"]).astype(int)
plot_data = plot_data.sort_values(["year", "topic_type"])

# Use consistent topic colors and ordering across both charts.
topics = plot_data["topic_type"].drop_duplicates().tolist()
palette = px.colors.qualitative.T10
color_map = {
    topic: palette[i % len(palette)]
    for i, topic in enumerate(topics)
}


def make_interactive_chart(value, title, ylabel):
    fig = px.bar(
        plot_data,
        x="year",
        y=value,
        color="topic_type",
        barmode="stack",
        title=title,
        labels={
            "year": "Year",
            "topic_type": "Topic Type",
            "n": "Number of segments",
            "proportion": "Share of segments (%)",
        },
        category_orders={"topic_type": topics},
        color_discrete_map=color_map,
        hover_data={
            "year": True,
            "n": ":,.0f",
            "proportion": ":.1f",
        },
        template="plotly_white",
        height=500,
    )

    # Numeric year axis: show a tick every two calendar years.
    fig.update_xaxes(
        tickmode="linear",
        tick0=int(plot_data["year"].min()),
        dtick=2,
        tickformat="d",
        tickangle=-45,
        tickfont=dict(size=10),
        title="Year",
    )

    fig.update_yaxes(title=ylabel)
    fig.update_layout(
        legend_title_text="Topic Type",
        bargap=0.15,
    )

    if value == "proportion":
        fig.update_yaxes(range=[0, 100], ticksuffix="%")

    return fig


fig_topic_counts = make_interactive_chart(
    "n",
    "Topic Distribution Over Time",
    "Number of segments",
)

fig_topic_counts.show()


In [ ]:
fig_topic_props = make_interactive_chart(
    "proportion",
    "Distribution of Topics Over Time",
    "Share of segments (%)",
)

fig_topic_props.show()

In [ ]:
# Interactive Line Chart for Topics
fig_topic = px.line(
    plot_data,
    x="year",
    y="proportion",
    color="topic_type",
    line_dash="topic_type",
    title="Topic-Type Trends Over Time",
    labels={
        "proportion": "Share of topic assignments (%)",
        "year": "Year",
        "topic_type": "Topic type"
    },
    color_discrete_map=color_map,
    category_orders={"topic_type": topics},
    markers=True,
    template="plotly_white",
    height=500
)

fig_topic.show()

In [ ]:
# 1. Group by year, relevance, and topic_type
topic_relevance_year = (
    df_topic.dropna(subset=['year', 'relevance_to_original_topic', 'topic_type'])
    .groupby(['year', 'relevance_to_original_topic', 'topic_type'])
    .size()
    .reset_index(name='n')
)

# 2. Calculate topic proportion per year within each relevance group
topic_relevance_year['proportion'] = (
    topic_relevance_year
    .groupby(['year', 'relevance_to_original_topic'])['n']
    .transform(lambda x: x / x.sum())
)

# 3. Faceted Interactive Line Chart using Plotly
fig_topic_relevance = px.line(
    topic_relevance_year,
    x='year',
    y='proportion',
    color='topic_type',
    line_dash='topic_type',
    facet_col='relevance_to_original_topic',
    facet_col_wrap=2,  # Wraps subplots if there are many relevance categories
    markers=True,
    title="Topic-Type Trends Over Time by Relevance Category",
    labels={
        'proportion': 'Share of topic assignments within relevance',
        'year': 'Year',
        'topic_type': 'Topic type',
        'relevance_to_original_topic': 'Relevance'
    }
)

# Clean up facet titles for a tidier display
fig_topic_relevance.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig_topic_relevance.show()

In [ ]:
# Optional: save standalone interactive files to open in a browser.
fig_topic_counts.write_html("figures/topic_counts_interactive.html")
fig_topic_props.write_html("figures/topic_proportions_interactive.html")
fig_topic.write_html("figures/topic_yearly_interactive.html")
fig_topic_relevance.write_html("figures/topic_relevance_yearly_interactive.html")

## 2.3 Gender Mentions Over Time

In [ ]:
# Aggregation
gender_year = (
    df.dropna(subset=['year', 'gender_mentioned'])
    .groupby(['year', 'gender_mentioned'])
    .size()
    .reset_index(name='n')
)
gender_year['proportion'] = gender_year.groupby('year')['n'].transform(lambda x: (x / x.sum()) * 100)

# Start from your aggregated gender_year table.
plot_data = gender_year.copy()
plot_data["year"] = pd.to_numeric(plot_data["year"]).astype(int)
plot_data = plot_data.sort_values(["year", "gender_mentioned"])

# Use consistent topic colors and ordering across both charts.
gender = plot_data["gender_mentioned"].drop_duplicates().tolist()
palette = px.colors.qualitative.T10
color_map = {
    topic: palette[i % len(palette)]
    for i, topic in enumerate(gender)
}

In [ ]:
# Interactive bar chart based on counts

def make_interactive_chart(value, title, ylabel):
    fig = px.bar(
        plot_data,
        x="year",
        y=value,
        color="gender_mentioned",
        barmode="stack",
        title=title,
        labels={
            "year": "Year",
            "gender_mentioned": "Gender Mentioned",
            "n": "Number of segments",
            "proportion": "Share of segments (%)",
        },
        category_orders={"gender_mentioned": gender},
        color_discrete_map=color_map,
        hover_data={
            "year": True,
            "n": ":,.0f",
            "proportion": ":.1f",
        },
        template="plotly_white",
        height=500,
    )

    # Numeric year axis: show a tick every two calendar years.
    fig.update_xaxes(
        tickmode="linear",
        tick0=int(plot_data["year"].min()),
        dtick=2,
        tickformat="d",
        tickangle=-45,
        tickfont=dict(size=10),
        title="Year",
    )

    fig.update_yaxes(title=ylabel)
    fig.update_layout(
        legend_title_text="Gender Mentioned",
        bargap=0.15,
    )

    if value == "proportion":
        fig.update_yaxes(range=[0, 100], ticksuffix="%")

    return fig


fig_gender_counts = make_interactive_chart(
    "n",
    "Distribution of Gender Mentions Over Time",
    "Number of segments",
)

fig_gender_counts.show()

In [ ]:
# Gender Mentions Composition Over Time (Proportions)

fig_gender_props = make_interactive_chart(
    "proportion",
    "Distribution of Gender Mentions Over Time",
    "Share of segments (%)",
)

fig_gender_props.show()

In [ ]:
# Interactive line charts

fig_gender_time = px.line(
    gender_year,
    x='year',
    y='proportion',
    color='gender_mentioned',
    markers=True,
    title="Changing Gender Composition Over Time",
    labels={'proportion': 'Share of segments (%)', 'year': 'Year', 'gender_mentioned': 'Gender Mentioned'}
)
fig_gender_time.show()

In [ ]:
# Optional: save standalone interactive files to open in a browser.
fig_gender_counts.write_html("figures/gender_counts_interactive.html")
fig_gender_props.write_html("figures/gender_proportions_interactive.html")
fig_gender_time.write_html("figures/gender_yearly_interactive.html")


# 3. Bivariate Analysis: 
## 3.1 Topic Type × Relevance

### Contingency Table, Statistical Testing & Effect Size
Construct a contingency matrix, compute row percentages, execute Chi-Square / Fisher Exact tests, and compute Cramer's V for effect size measurement.

In [ ]:
# Helper function for Cramer's V computation
def cramers_v(contingency_matrix):
    chi2 = chi2_contingency(contingency_matrix)[0]
    n = contingency_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = contingency_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1)) / (n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

# Cross-tabulation
tab_topic_relevance = pd.crosstab(df_topic['topic_type'], df_topic['relevance_to_original_topic'])
print("--- Contingency Table ---")
print(tab_topic_relevance)

print("\n--- Row Percentages ---")
print(tab_topic_relevance.div(tab_topic_relevance.sum(axis=1), axis=0))

# Chi-Square Test
chi2, p, dof, expected = chi2_contingency(tab_topic_relevance)
print(f"\nChi-Square Statistic: {chi2:.4f}, p-value: {p:.4e}, DoF: {dof}")

print("\nExpected Frequencies:")
print(pd.DataFrame(expected, index=tab_topic_relevance.index, columns=tab_topic_relevance.columns))

# Effect Size
v_val = cramers_v(tab_topic_relevance)
print(f"\nCramer's V: {v_val:.4f}")

### Proportional Heatmap & Standardized Residuals
Plot the conditional distribution percentage heatmap alongside standardized residuals to highlight over- and under-represented cells.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def annotated_heatmap(data, fmt, title, cmap, center=None):
    fig, ax = plt.subplots(figsize=(10, 6))

    sns.heatmap(
        data,
        annot=False,
        cmap=cmap,
        center=center,
        linewidths=0.5,
        ax=ax,
    )

    # Add labels explicitly to every finite cell.
    mesh = ax.collections[0]
    for row in range(data.shape[0]):
        for col in range(data.shape[1]):
            value = data.iloc[row, col]
            if not np.isfinite(value):
                continue

            # Choose a readable text color for each cell's background.
            red, green, blue, _ = mesh.cmap(mesh.norm(value))
            brightness = 0.299 * red + 0.587 * green + 0.114 * blue

            ax.text(
                col + 0.5,
                row + 0.5,
                format(value, fmt),
                ha="center",
                va="center",
                color="black" if brightness > 0.55 else "white",
                fontsize=10,
            )

    ax.set_title(title)
    ax.set_xlabel("Relevance to original topic")
    ax.set_ylabel("Topic type")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    fig.tight_layout()
    plt.show()


# 1. Proportional Heatmap
# Rows with no observations have undefined percentages.
row_totals = tab_topic_relevance.sum(axis=1).replace(0, np.nan)
topic_relevance_prop = tab_topic_relevance.div(row_totals, axis=0)

hm1 = annotated_heatmap(
    topic_relevance_prop,
    fmt=".1%",
    cmap="Blues",
    title=(
        "Relationship Between Topic Type and Relevance\n"
        "(Percentage distribution within each topic type)"
    ),
)

hm1

In [ ]:
# 2. Chi-Square Standardized Residuals Heatmap
observed = tab_topic_relevance.to_numpy(dtype=float)
row_sums = observed.sum(axis=1)
col_sums = observed.sum(axis=0)
n_total = observed.sum()

if n_total <= 0:
    raise ValueError("The table must contain observations.")

r_prob = row_sums / n_total
c_prob = col_sums / n_total

expected_arr = np.outer(row_sums, col_sums) / n_total
denominator = np.sqrt(
    expected_arr * np.outer(1 - r_prob, 1 - c_prob)
)

# Residuals are undefined wherever the denominator is zero.
std_residuals = np.divide(
    observed - expected_arr,
    denominator,
    out=np.full_like(expected_arr, np.nan),
    where=denominator > 0,
)

std_res_df = pd.DataFrame(
    std_residuals,
    index=tab_topic_relevance.index,
    columns=tab_topic_relevance.columns,
)

hm2 = annotated_heatmap(
    std_res_df,
    fmt=".2f",
    cmap="vlag",
    center=0,
    title=(
        "Topic Type × Relevance: Chi-Square Standardized Residuals\n"
        "(Positive = over-observed; Negative = under-observed)"
    ),
)


hm2

## 3.2 Gender Mentioned × Relevance
Run contingency analysis, hypothesis tests, and standard residual heatmaps for gender mentions against relevance.

### Contingency Table, Statistical Testing, & Effect Size 

In [ ]:
# Cross-tabulation
tab_gender_relevance = pd.crosstab(df['gender_mentioned'], df['relevance_to_original_topic'])
print("--- Contingency Table ---")
print(tab_gender_relevance)

print("\n--- Row Percentages ---")
print(tab_gender_relevance.div(tab_gender_relevance.sum(axis=1), axis=0))

# Chi-Square Test
chi2, p, dof, expected = chi2_contingency(tab_gender_relevance)
print(f"\nChi-Square Statistic: {chi2:.4f}, p-value: {p:.4e}, DoF: {dof}")

print("\nExpected Frequencies:")
print(pd.DataFrame(expected, index=tab_gender_relevance.index, columns=tab_gender_relevance.columns))

# Effect Size
v_val = cramers_v(tab_gender_relevance)
print(f"\nCramer's V: {v_val:.4f}")

### Proportional Heatmap & Standardized Residuals
Plot the conditional distribution percentage heatmap alongside standardized residuals to highlight over- and under-represented cells.

In [ ]:
# Proportions within each row; empty rows remain undefined.
gender_relevance_prop = tab_gender_relevance.div(
    tab_gender_relevance.sum(axis=1).replace(0, np.nan),
    axis=0,
)

fig, ax = plt.subplots(figsize=(8, 4))

sns.heatmap(
    gender_relevance_prop,
    annot=False,
    cmap="Blues",
    linewidths=0.5,
    ax=ax,
)

# Explicitly annotate every valid cell.
mesh = ax.collections[0]

for row in range(gender_relevance_prop.shape[0]):
    for col in range(gender_relevance_prop.shape[1]):
        value = gender_relevance_prop.iloc[row, col]

        if not np.isfinite(value):
            continue

        red, green, blue, _ = mesh.cmap(mesh.norm(value))
        brightness = 0.299 * red + 0.587 * green + 0.114 * blue

        ax.text(
            col + 0.5,
            row + 0.5,
            f"{value:.1%}",
            ha="center",
            va="center",
            color="black" if brightness > 0.55 else "white",
            fontsize=10,
        )

ax.set_title("Relationship Between Gender Mention and Relevance")
ax.set_xlabel("Relevance to original topic")
ax.set_ylabel("Gender mentioned")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()

# Optional: save before displaying.
# from pathlib import Path
# Path("figures").mkdir(parents=True, exist_ok=True)
# fig.savefig(
#     "figures/heatmap_gender_relevance_proportions.png",
#     dpi=300,
#     bbox_inches="tight",
# )

plt.show()

In [ ]:
# 2. Chi-Square Standardized Residuals Heatmap
observed = tab_gender_relevance.to_numpy(dtype=float)
row_sums = observed.sum(axis=1)
col_sums = observed.sum(axis=0)
n_total = observed.sum()

if n_total <= 0:
    raise ValueError("The table must contain observations.")

r_prob = row_sums / n_total
c_prob = col_sums / n_total

expected_arr = np.outer(row_sums, col_sums) / n_total
denominator = np.sqrt(
    expected_arr * np.outer(1 - r_prob, 1 - c_prob)
)

# Residuals are undefined wherever the denominator is zero.
std_residuals = np.divide(
    observed - expected_arr,
    denominator,
    out=np.full_like(expected_arr, np.nan),
    where=denominator > 0,
)

std_res_df = pd.DataFrame(
    std_residuals,
    index=tab_gender_relevance.index,
    columns=tab_gender_relevance.columns,
)

hm2 = annotated_heatmap(
    std_res_df,
    fmt=".2f",
    cmap="vlag",
    center=0,
    title=(
        "Gender × Relevance: Chi-Square Standardized Residuals\n"
        "(Positive = over-observed; Negative = under-observed)"
    ),
)


hm2

## 3.3. Gender Mentioned × Topic Type 

### Contingency Table, Statistical Testing & Effect Size
Construct a contingency matrix, compute row percentages, execute Chi-Square / Fisher Exact tests, and compute Cramer's V for effect size measurement.

In [ ]:
# Cross-tabulation
tab_gender_topic = pd.crosstab(df_topic['gender_mentioned'], df_topic['topic_type'])
print("--- Contingency Table ---")
print(tab_gender_topic)

print("\n--- Row Percentages ---")
print(tab_gender_topic.div(tab_gender_topic.sum(axis=1), axis=0))

# Chi-Square Test
chi2, p, dof, expected = chi2_contingency(tab_gender_topic)
print(f"\nChi-Square Statistic: {chi2:.4f}, p-value: {p:.4e}, DoF: {dof}")

print("\nExpected Frequencies:")
print(pd.DataFrame(expected, index=tab_gender_topic.index, columns=tab_gender_topic.columns))

# Effect Size
v_val = cramers_v(tab_gender_topic)
print(f"\nCramer's V: {v_val:.4f}")

### Proportional Heatmap & Standardized Residuals
Plot the conditional distribution percentage heatmap alongside standardized residuals to highlight over- and under-represented cells.

In [ ]:
# Proportions within each row; empty rows remain undefined.
gender_topic_prop = tab_gender_topic.div(
    tab_gender_topic.sum(axis=1).replace(0, np.nan),
    axis=0,
)

# Transpose the matrix
gender_topic_prop = gender_topic_prop.T

fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    gender_topic_prop,
    annot=False,
    cmap="Blues",
    linewidths=0.5,
    ax=ax,
)

# Explicitly annotate every valid cell.
mesh = ax.collections[0]

for row in range(gender_topic_prop.shape[0]):
    for col in range(gender_topic_prop.shape[1]):
        value = gender_topic_prop.iloc[row, col]

        if not np.isfinite(value):
            continue

        red, green, blue, _ = mesh.cmap(mesh.norm(value))
        brightness = 0.299 * red + 0.587 * green + 0.114 * blue

        ax.text(
            col + 0.5,
            row + 0.5,
            f"{value:.1%}",
            ha="center",
            va="center",
            color="black" if brightness > 0.55 else "white",
            fontsize=10,
        )

ax.set_title("Relationship Between Gender Mention and Topic Type")
ax.set_xlabel("Gender mentioned")
ax.set_ylabel("Topic Type")

plt.setp(ax.get_xticklabels(), rotation=0, ha="center")
plt.setp(ax.get_yticklabels(), rotation=0)

fig.tight_layout()
plt.show()

In [ ]:
# 2. Chi-Square Standardized Residuals Heatmap
observed = tab_gender_topic.to_numpy(dtype=float)
row_sums = observed.sum(axis=1)
col_sums = observed.sum(axis=0)
n_total = observed.sum()

if n_total <= 0:
    raise ValueError("The table must contain observations.")

r_prob = row_sums / n_total
c_prob = col_sums / n_total

expected_arr = np.outer(row_sums, col_sums) / n_total
denominator = np.sqrt(
    expected_arr * np.outer(1 - r_prob, 1 - c_prob)
)

# Residuals are undefined wherever the denominator is zero.
std_residuals = np.divide(
    observed - expected_arr,
    denominator,
    out=np.full_like(expected_arr, np.nan),
    where=denominator > 0,
)

std_res_df = pd.DataFrame(
    std_residuals,
    index=tab_gender_topic.index,
    columns=tab_gender_topic.columns,
)

hm2 = annotated_heatmap(
    std_res_df.T,  # Transpose 
    fmt=".2f",
    cmap="vlag",
    center=0,
    title=(
        "Gender × Topic Type: Chi-Square Standardized Residuals\n"
        "(Positive = over-observed; Negative = under-observed)"
    ),
)


hm2

# 4. Gender, Relevance, and Topic Over Time

## 4.1 Gender & Relevance

In [ ]:
# Prepare data and keep colors consistent across both figures.
gender_data = df.dropna(subset=["year", "gender_mentioned"]).copy()
gender_data["year"] = pd.to_numeric(
    gender_data["year"], errors="coerce"
)
gender_data = gender_data.dropna(subset=["year"])
gender_data["year"] = gender_data["year"].astype(int)

gender_categories = sorted(
    gender_data["gender_mentioned"].unique(), key=str
)
palette = px.colors.qualitative.Plotly
gender_colors = {
    category: palette[i % len(palette)]
    for i, category in enumerate(gender_categories)
}


def plot_gender_by_facet(data, facet, facet_label, title):
    subset = data.dropna(subset=[facet]).copy()

    if subset.empty:
        raise ValueError(f"No observations available for {facet}.")

    facet_categories = sorted(subset[facet].unique(), key=str)
    years = range(subset["year"].min(), subset["year"].max() + 1)

    counts = subset.groupby(
        ["year", facet, "gender_mentioned"],
        observed=True,
    ).size()

    # Include every gender category for each year and panel.
    full_index = pd.MultiIndex.from_product(
        [years, facet_categories, gender_categories],
        names=["year", facet, "gender_mentioned"],
    )

    yearly = (
        counts.reindex(full_index, fill_value=0)
        .rename("n")
        .reset_index()
    )

    totals = yearly.groupby(["year", facet])["n"].transform("sum")

    # Years with no observations in a panel remain undefined.
    yearly["proportion"] = (
        yearly["n"].div(totals.where(totals > 0)).mul(100)
    )

    fig = px.line(
        yearly,
        x="year",
        y="proportion",
        color="gender_mentioned",
        facet_col=facet,
        facet_col_wrap=2,
        markers=True,
        color_discrete_map=gender_colors,
        category_orders={
            "gender_mentioned": gender_categories,
            facet: facet_categories,
        },
        hover_data={
            "n": True,
            "proportion": ":.1f",
        },
        labels={
            "year": "Year",
            "proportion": "Share of segments (%)",
            "gender_mentioned": "Gender Mentioned",
            facet: facet_label,
            "n": "Number of segments",
        },
        title=title,
        template="plotly_white",
        height=300 * ((len(facet_categories) + 2) // 3) + 150,
    )

    fig.update_xaxes(
        tickmode="linear",
        tick0=min(years),
        dtick=2,
        tickformat="d",
        tickangle=-45,
        tickfont=dict(size=10),
    )

    fig.update_yaxes(
        range=[0, 100],
        ticksuffix="%",
    )

    # Leave gaps where a panel has no observations.
    fig.update_traces(connectgaps=False)

    # Simplify panel headings.
    fig.for_each_annotation(
        lambda annotation: annotation.update(
            text=annotation.text.split("=", 1)[-1]
        )
    )

    return fig

In [ ]:
# 1. Gender mention distribution by relevance
fig_gender_relevance = plot_gender_by_facet(
    gender_data,
    facet="relevance_to_original_topic",
    facet_label="Relevance",
    title="Gender Mention Distribution Over Time, by Relevance",
)
fig_gender_relevance.show()


In [ ]:
# Share of segments mentioning gender (True/Yes) by year and relevance_to_original_topic.
# Exclude missing gender labels, matching the original notebook's denominator.
base = df.dropna(subset=['year', 'relevance_to_original_topic', 'gender_mentioned']).copy()
base['year'] = pd.to_numeric(base['year'], errors='coerce')
base = base.dropna(subset=['year'])
base['year'] = base['year'].astype(int)
# Supports both newer boolean True/False and older string Yes/No datasets.
base['mentions_gender'] = base['gender_mentioned'].astype('string').str.strip().str.lower().isin(['true', 'yes', '1'])
gender_relevance_share = (
    base.groupby(['year', 'relevance_to_original_topic'], observed=True)
    .agg(n=('mentions_gender', 'size'), n_gender=('mentions_gender', 'sum'))
    .reset_index()
)
gender_relevance_share['proportion_gender'] = 100 * gender_relevance_share['n_gender'] / gender_relevance_share['n']
print(gender_relevance_share[['year', 'relevance_to_original_topic', 'n', 'n_gender', 'proportion_gender']].head(12))
gender_relevance_facets = px.line(
    gender_relevance_share.sort_values('year'), x='year', y='proportion_gender',
    facet_col='relevance_to_original_topic', facet_col_wrap=2, markers=True,
    title='Share of Segments Mentioning Gender by Relevance Over Time',
    labels={'proportion_gender': 'Segments mentioning gender (%)', 'year': 'Year'},
    hover_data={'n': True, 'n_gender': True, 'proportion_gender': ':.1f'},
    template='plotly_white',
)
gender_relevance_facets.update_yaxes(ticksuffix='%')
gender_relevance_facets.update_traces(connectgaps=False)
gender_relevance_facets.show()


## 4.2 Gender and Topic Over Time

In [ ]:
# 2. Gender mention distribution by topic
gender_topic_data = df_topic.dropna(
    subset=["year", "gender_mentioned", "topic_type"]
).copy()

gender_topic_data["year"] = pd.to_numeric(
    gender_topic_data["year"], errors="coerce"
)
gender_topic_data = gender_topic_data.dropna(subset=["year"])
gender_topic_data["year"] = gender_topic_data["year"].astype(int)

fig_gender_topic = plot_gender_by_facet(
    gender_topic_data,
    facet="topic_type",
    facet_label="Topic",
    title="Gender Mention Distribution Over Time, by Topic",
)
fig_gender_topic.show()

In [ ]:
# Share of segments mentioning gender (True/Yes) by year and topic_type.
# Exclude missing gender labels, matching the original notebook's denominator.
base = df_topic.dropna(subset=['year', 'topic_type', 'gender_mentioned']).copy()
base['year'] = pd.to_numeric(base['year'], errors='coerce')
base = base.dropna(subset=['year'])
base['year'] = base['year'].astype(int)
# Supports both newer boolean True/False and older string Yes/No datasets.
base['mentions_gender'] = base['gender_mentioned'].astype('string').str.strip().str.lower().isin(['true', 'yes', '1'])
gender_topic_share = (
    base.groupby(['year', 'topic_type'], observed=True)
    .agg(n=('mentions_gender', 'size'), n_gender=('mentions_gender', 'sum'))
    .reset_index()
)
gender_topic_share['proportion_gender'] = 100 * gender_topic_share['n_gender'] / gender_topic_share['n']
print(gender_topic_share[['year', 'topic_type', 'n', 'n_gender', 'proportion_gender']].head(12))
gender_topic_facets = px.line(
    gender_topic_share.sort_values('year'), x='year', y='proportion_gender',
    facet_col='topic_type', facet_col_wrap=2, markers=True,
    title='Share of Segments Mentioning Gender by Topic Over Time',
    labels={'proportion_gender': 'Segments mentioning gender (%)', 'year': 'Year'},
    hover_data={'n': True, 'n_gender': True, 'proportion_gender': ':.1f'},
    template='plotly_white',
)
gender_topic_facets.update_yaxes(ticksuffix='%')
gender_topic_facets.update_traces(connectgaps=False)
gender_topic_facets.show()


In [ ]:
# Optional: save standalone interactive files to open in a browser.
fig_gender_time.write_html("figures/gender_time.html")
gender_relevance_facets.write_html("figures/gender_relevance_facets.html")
gender_topic_facets.write_html("figures/gender_topic_facets.html")